# CRF（條件隨機場，2001）

- **核心原理**：序列標註的條件機率
- **變體**：BiLSTM-CRF（神經網路產生分數、CRF 負責「整條序列」的一致性）；semi-Markov CRF；一般圖結構的 CRF
- **推薦資料集**：CoNLL-2003（英文新聞的命名實體辨識，NER）
- **實作**：✋ PyTorch 手刻 linear-chain CRF；🧰 `sklearn-crfsuite` 對照

第 20 章的 HMM 標註器是**生成式**：學 $P(x, y) = P(y)\,P(x \mid y)$，發射機率只能看「這個字本身」。結果 test F1 只有約 0.73，主要錯在沒看過的字（OOV）。要用「字形」「字尾」「前後的字」這些**互相重疊**的線索，HMM 會重複計算（第六章樸素貝氏的問題）。

CRF（Lafferty, McCallum & Pereira 2001）是**判別式**：直接寫
$$P(y_{1:T} \mid x) = \frac{\exp\big(\text{score}(x, y)\big)}{Z(x)}, \qquad \text{score}(x, y) = \sum_{t}\underbrace{e_t(y_t)}_{\text{發射分數：任意特徵}} + \sum_{t > 1}\underbrace{A_{y_{t-1} y_t}}_{\text{轉移分數}}$$

- 發射分數 $e_t(k)$ 可以是任何特徵的加權和（像第三章的邏輯回歸），特徵可以重疊、可以看整句話；
- 轉移分數 $A$ 讓模型學「O 後面不能接 I-PER」這種**輸出之間**的關係；
- $Z(x)$ = 所有 $K^T$ 種標註的 $\exp(\text{score})$ 總和 —— 用第 20 章的 forward 演算法算（只是把機率換成分數）。

一句話：**CRF = 每個字一個邏輯回歸 + 標籤之間的轉移分數，對整條序列一起做 softmax。**

這一章的重點：**gold 路徑分數、log-partition（forward）與 Viterbi 的 PyTorch 批次版、padding 與 mask、為什麼要對整條序列正規化（vs 每個字各自 softmax）、特徵工程的效果、轉移矩陣學到了什麼、以及跟 BiLSTM-CRF 的關係。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3C、4、5 節開頭另有各自的符號表或名詞表。

**資料**：`load_conll2003()` 回傳 Hugging Face 的 `DatasetDict`（train 14,041 句、validation 3,250 句、test 3,453 句）與標籤名稱。每句話：`tokens`（字的清單）、`ner_tags`（0–8 的整數）。標籤 9 種：`O` 與 {`B-`, `I-`} × {PER 人名, ORG 組織, LOC 地點, MISC 其他}。`B-` = 實體的第一個字、`I-` = 後續字。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $T$、$K$ | 句子長度、標籤數 | 1–124、9 |
| $B$ | 一個 batch 的句子數 | 64 |
| $M$ | 每個字的特徵數 | 依特徵組合（完整版 12） |
| $E$ | 發射分數 $e_t(k)$ | $(B, T, K)$ |
| $A$ | 轉移分數（$A_{jk}$：從 $j$ 到 $k$） | $(K, K)$ |
| $s$、$f$ | 開始、結束分數（句首是 $k$、句尾是 $k$ 的加分） | $(K,)$ |
| mask | 哪些位置是真的字（不是 padding） | $(B, T)$ bool |
| $\ln Z$ | log-partition | $(B,)$ |
| NLL | $\ln Z - \text{score}(x, y_{\text{gold}})$ | 純量（batch 平均） |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第三章 | 邏輯回歸 | Softmax 多分類：$P(y = k \mid x) = e^{w_k \cdot x} / \sum_j e^{w_j \cdot x}$；最小化負 log-likelihood（交叉熵）；判別式模型。L2 正則化防止過擬合。 |
| 第六章 | 樸素貝氏 | 生成式模型；特徵獨立假設 → 重疊的特徵被重複計算；log-sum-exp。 |
| 第 20 章 | HMM（`01-classic-ml/20-hmm`），同一份 CoNLL-2003 | forward（$O(TK^2)$、log 空間）、Viterbi（max + 回溯）；監督式 HMM 用數數 + 字形 UNK：test F1 ≈ 0.73（precision 0.76、recall 0.70）；test 的實體字約 31% 在 train 沒出現過（所有字是 12%）；成績在 `results/conll2003_ner.csv`。 |
| 第三部分 | RNN／LSTM | BiLSTM-CRF 的 BiLSTM（這裡先當黑盒子用）。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. Softmax 與交叉熵**（第三章）：$P(k) = e^{s_k}/\sum_j e^{s_j}$，$-\ln P(\text{gold}) = \ln\sum_j e^{s_j} - s_{\text{gold}}$ = 「log-partition − 正確答案的分數」。CRF 的損失長得一模一樣，只是「類別」變成「整條標註序列」。

**2. forward（分數版）**（第 20 章 3A）：HMM 的 $\ln\alpha_t(k) = \text{logsumexp}_j(\ln\alpha_{t-1}(j) + \ln A_{jk}) + \ln B_{k, x_t}$。CRF 把 $\ln A$、$\ln B$ 換成**沒有正規化的分數** $A$、$e_t$：
$$\alpha_1 = s + e_1, \qquad \alpha_t(k) = \text{logsumexp}_j\big(\alpha_{t-1}(j) + A_{jk}\big) + e_t(k), \qquad \ln Z = \text{logsumexp}_k\big(\alpha_T(k) + f_k\big)$$

**3. Viterbi**（第 20 章 3C）：logsumexp 換成 max，記回溯指標。

**4. 梯度 = 期望特徵 − 觀測特徵**：$\partial \ln Z / \partial e_t(k) = P(y_t = k \mid x)$（邊際機率）。所以 NLL 對 $e_t(k)$ 的梯度 = 「模型認為 $t$ 是 $k$ 的機率」−「gold 是不是 $k$」—— 跟邏輯回歸的「預測 − 標籤」一樣。PyTorch 的 autograd 會自動算，不用手寫 backward。

#### 程式（PyTorch，第一次正式用）

**5. 張量與 autograd**：`torch.tensor`、`requires_grad`；`loss.backward()` 算梯度、`optimizer.step()` 更新、`optimizer.zero_grad()` 清掉。
**6. `nn.EmbeddingBag(n_features, K, mode="sum")`**：輸入一堆特徵編號，輸出「這些特徵的權重加總」—— 就是稀疏特徵的線性模型（邏輯回歸的 $w \cdot x$，$x$ 是 0/1）。`padding_idx=0` 的特徵權重固定為 0。
**7. padding 與 mask**：一個 batch 的句子長度不同 → 補到一樣長，用 `mask[b, t] = True` 標記真的字。forward 時 padding 的位置要「什麼都不做」：`torch.where(mask[:, t, None], 新值, 舊值)`。
**8. `gather`**：`E.gather(2, y[:, :, None])` 取出每個位置「gold 標籤」那一欄的分數，$(B, T, K) \to (B, T, 1)$。
**9. `torch.logsumexp(x, dim)`**、`x.max(dim)` 回傳 `(values, indices)`。

---
## 1. 資料與探索性資料分析（EDA）

In [ ]:
import itertools
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from ml_dl_llm.data import load_conll2003

torch.manual_seed(0)
ds, label_names = load_conll2003()
K = len(label_names)
splits = {name: ([list(ex["tokens"]) for ex in ds[name]], [list(ex["ner_tags"]) for ex in ds[name]]) for name in ["train", "validation", "test"]}
train_tokens, train_tags = splits["train"]
print("標籤：", label_names)
print({name: len(toks) for name, (toks, _) in splits.items()}, "句")
print("一句話的例子：", list(zip(train_tokens[0], [label_names[t] for t in train_tags[0]])))


def entities(tags: list[str]) -> list[tuple[str, int, int]]:
    """BIO 標籤 → [(類型, 開始, 結束（不含）), ...]。"""
    out, start, typ = [], None, None
    for i, t in enumerate(tags + ["O"]):
        if start is not None and not t.startswith("I-" + typ):
            out.append((typ, start, i))
            start = None
        if t.startswith("B-") or (t.startswith("I-") and start is None):
            start, typ = i, t[2:]
    return out


rows = []
for name, (toks, tags) in splits.items():
    ents = [e for y in tags for e in entities([label_names[t] for t in y])]
    c = Counter(e[0] for e in ents)
    rows.append({"split": name, "sentences": len(toks), "tokens": sum(map(len, toks)), "entities": len(ents), **{k: c[k] for k in ["PER", "ORG", "LOC", "MISC"]},
                 "mean entity length": np.mean([e[2] - e[1] for e in ents])})
display(pd.DataFrame(rows).set_index("split").round(2))

In [ ]:
word_count = Counter(w for s in train_tokens for w in s)
test_tokens, test_tags = splits["test"]
ent_words = [w for s, y in zip(test_tokens, test_tags) for w, t in zip(s, y) if t != 0]
print(f"test 的字 OOV 比例：{np.mean([word_count[w] == 0 for s in test_tokens for w in s]):.1%}；實體字 OOV 比例：{np.mean([word_count[w] == 0 for w in ent_words]):.1%}")

lens = [len(s) for s in train_tokens]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].hist(lens, bins=60)
axes[0].set_xlabel("sentence length (train)")
amb = Counter()
tag_of = {}
for s, y in zip(train_tokens, train_tags):
    for w, t in zip(s, y):
        tag_of.setdefault(w, Counter())[label_names[t][2:] if t else "O"] += 1
ambiguous = {w: c for w, c in tag_of.items() if len(c) > 1 and sum(c.values()) >= 20}
top_amb = sorted(ambiguous.items(), key=lambda kv: -sum(kv[1].values()))[:12]
pd.DataFrame({w: dict(c) for w, c in top_amb}).fillna(0).T.plot.barh(stacked=True, ax=axes[1])
axes[1].set_title("frequent words with more than one entity type")
plt.show()
print(f"最長的句子：{max(lens)} 個字；中位數 {int(np.median(lens))}；只有 1 個字的句子：{sum(l == 1 for l in lens)} 句")

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 四種實體各有幾個？平均一個實體幾個字？B- 和 I- 的數量差多少？
# 答：
#
# (2) 有歧義的字（同一個字有時是 LOC、有時是 ORG）：舉兩個例子，人類靠什麼判斷？（提示：前後文 —— "Germany beat ..." 的 Germany 是隊伍 = ORG）
#     HMM 的發射機率 P(字 | 標籤) 能用上前後文嗎？
# 答：
#
# (3) 句長從 1 到 100 多：一個 batch 裡長短混在一起，padding 會浪費多少計算？怎麼減少？（提示：依長度排序再切 batch）
# 答：

---
## 2. 特徵與 Baseline：每個字各自 softmax（沒有轉移）

### 特徵

每個字轉成一串「特徵字串」，例如 `"w=german"`、`"shape=Xx"`、`"suf3=man"`、`"-1:w=rejects"`。所有在 train 出現過的特徵字串編號 → `EmbeddingBag` 每個特徵一組 $K$ 個權重，加起來就是發射分數 $e_t$（前置複習 6）。

| 特徵 | 例子（"German"） | 為什麼有用 |
| --- | --- | --- |
| `bias` | 每個字都有 | 標籤的先驗 |
| `w=` 小寫字 | `w=german` | 字本身 |
| `shape=` 字形 | `shape=Xx` | 大寫開頭 → 可能是實體（第 20 章的 UNK:字形） |
| `istitle`／`isupper`／`isdigit` | `istitle=1` | 同上 |
| `pre3=`、`suf3=` | `pre3=ger`、`suf3=man` | 字尾 -land、-ville、-son、-ese |
| `-1:w=`、`-1:shape=` | `-1:w=rejects` | 前一個字（"Mr." 後面是人名） |
| `+1:w=`、`+1:shape=` | `+1:w=call` | 後一個字（"... said" 前面常是人名） |

句首的「前一個字」用 `<BOS>`、句尾的「後一個字」用 `<EOS>`，讓每個字的特徵數都一樣（$M$ 固定，才能疊成張量）。

### Baseline

1. 第 20 章的 HMM（成績已在 `conll2003_ner`）；
2. **同樣的特徵、每個字各自 softmax**（= 每個字一個邏輯回歸，第三章）：跟 CRF 唯一的差別是**沒有轉移分數、不對整條序列正規化**。

#### 最小例子：TODO 3（前後文特徵）

句子 `["EU", "rejects", "German", "call"]`，第 2 個字（"German"，索引 2）：

1. **正確答案**：前一個字 "rejects"（字形 `xx`：小寫連續縮成兩個）、後一個字 "call"（`xx`）→ `["-1:w=rejects", "-1:shape=xx", "+1:w=call", "+1:shape=xx"]`；第 0 個字（"EU"）沒有前一個字 → `["-1:w=<BOS>", "-1:shape=<BOS>", "+1:w=rejects", "+1:shape=xx"]`。
2. **積木**：`word_shape`、`toks[i - 1].lower()`。
3. **留給你**：寫進 `token_features` 的 `"context"` 部分。

In [ ]:
def word_shape(w: str) -> str:
    """大寫 X、小寫 x、數字 d；連續相同字元縮成兩個（第 20 章）。"""
    s = re.sub("[A-Z]", "X", w)
    s = re.sub("[a-z]", "x", s)
    s = re.sub("[0-9]", "d", s)
    return re.sub(r"(.)\1+", r"\1\1", s)


FEATURE_SETS = {
    "word": ["word"],
    "word+shape": ["word", "shape"],
    "word+shape+affix": ["word", "shape", "affix"],
    "full (+context)": ["word", "shape", "affix", "context"],
}


def token_features(toks: list[str], i: int, groups: list[str]) -> list[str]:
    """第 i 個字的特徵字串。groups 決定要哪幾組（每組的個數固定）。"""
    w = toks[i]
    f = ["bias"]
    if "word" in groups:
        f += ["w=" + w.lower()]
    if "shape" in groups:
        f += ["shape=" + word_shape(w), f"istitle={int(w.istitle())}", f"isupper={int(w.isupper())}", f"isdigit={int(w.isdigit())}"]
    if "affix" in groups:
        f += ["pre3=" + w[:3].lower(), "suf3=" + w[-3:].lower()]
    if "context" in groups:
        # TODO 3（程式）：前一個字、後一個字（各 2 個特徵：小寫字、字形）；沒有的話用 <BOS>／<EOS>
        raise NotImplementedError("TODO 3：前後文特徵")
    return f


toks_mini = ["EU", "rejects", "German", "call"]
print("字形：", [word_shape(w) for w in toks_mini])
# 寫完 TODO 3 之後，取消下面的註解來對照（應跟上面 markdown 的正確答案一樣）：
# print(token_features(toks_mini, 2, ["context"])[1:], "\n", token_features(toks_mini, 0, ["context"])[1:])

In [ ]:
class FeatureIndexer:
    """train 裡出現過的特徵字串 → 編號（從 1 開始；0 = 沒看過的特徵，權重固定 0）。"""

    def __init__(self, sents: list[list[str]], groups: list[str]) -> None:
        self.groups = groups
        feats = {f for s in sents for i in range(len(s)) for f in token_features(s, i, groups)}
        self.index = {f: j + 1 for j, f in enumerate(sorted(feats))}

    def encode(self, toks: list[str]) -> torch.Tensor:
        """(T, M) 的特徵編號。"""
        return torch.tensor([[self.index.get(f, 0) for f in token_features(toks, i, self.groups)] for i in range(len(toks))])


def encode_splits(groups: list[str]) -> tuple[FeatureIndexer, dict]:
    fx = FeatureIndexer(train_tokens, groups)
    data = {name: [(fx.encode(s), torch.tensor(y)) for s, y in zip(toks, tags)] for name, (toks, tags) in splits.items()}
    return fx, data


def batchify(items: list[tuple[torch.Tensor, torch.Tensor]]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """補 0 到同樣長度 → F (B, T, M)、Y (B, T)、mask (B, T)。"""
    T = max(len(y) for _, y in items)
    M = items[0][0].shape[1]
    F = torch.zeros(len(items), T, M, dtype=torch.long)
    Y = torch.zeros(len(items), T, dtype=torch.long)
    mask = torch.zeros(len(items), T, dtype=torch.bool)
    for b, (f, y) in enumerate(items):
        F[b, : len(y)], Y[b, : len(y)], mask[b, : len(y)] = f, y, True
    return F, Y, mask


t0 = time.perf_counter()
fx_full, data_full = encode_splits(FEATURE_SETS["full (+context)"])
print(f"完整特徵：{len(fx_full.index):,} 種特徵字串；每個字 M = {data_full['train'][0][0].shape[1]} 個；編碼花 {time.perf_counter() - t0:.0f} 秒")
F_ex, Y_ex, mask_ex = batchify(data_full["train"][:3])
print("一個 batch 的 shape：F", tuple(F_ex.shape), "、Y", tuple(Y_ex.shape), "、mask", tuple(mask_ex.shape), "；每句長度", mask_ex.sum(1).tolist())

#### 訓練與評估的共用工具

- `make_batches`：依長度排序後切成 64 句一批（減少 padding），每個 epoch 打亂「批」的順序。
- `train_model`：Adam；loss 加上 L2 懲罰 $\lambda\lVert W\rVert^2$（第三章）；每個 epoch 量 validation F1，**留下 validation 最好的那一個 epoch 的權重**（early stopping）。
- `evaluate`：`model.predict(F, mask)` 回傳每句話的標籤清單 → `ner_metrics`（實體層級 P／R／F1）。

模型只要有 `loss(F, Y, mask)`、`predict(F, mask)` 兩個方法就能共用。

In [ ]:
import copy

from ml_dl_llm.metrics import ner_metrics


def to_names(seqs: list[list[int]]) -> list[list[str]]:
    return [[label_names[t] for t in s] for s in seqs]


def make_batches(data: list, batch_size: int = 64) -> list[list[int]]:
    order = sorted(range(len(data)), key=lambda i: len(data[i][1]))
    return [order[i : i + batch_size] for i in range(0, len(order), batch_size)]


@torch.no_grad()
def predict_all(model: nn.Module, data: list, batch_size: int = 256) -> list[list[int]]:
    model.eval()
    out = [None] * len(data)
    for idx in make_batches(data, batch_size):
        F, _, mask = batchify([data[i] for i in idx])
        for i, p in zip(idx, model.predict(F, mask)):
            out[i] = p
    return out


def evaluate(model: nn.Module, data: list) -> dict:
    return ner_metrics(to_names([y.tolist() for _, y in data]), to_names(predict_all(model, data)))


def train_model(model: nn.Module, data: dict, epochs: int = 8, lr: float = 0.05, l2: float = 1e-6, seed: int = 0, verbose: bool = True) -> list[dict]:
    """回傳每個 epoch 的紀錄；結束時 model 的權重 = validation F1 最好的 epoch。"""
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    batches = make_batches(data["train"])
    rng = np.random.default_rng(seed)
    hist, best_f1, best_state = [], -1.0, None
    for ep in range(epochs):
        model.train()
        t0, total = time.perf_counter(), 0.0
        for b in rng.permutation(len(batches)):
            F, Y, mask = batchify([data["train"][i] for i in batches[b]])
            loss = model.loss(F, Y, mask) + l2 * sum((p**2).sum() for p in model.parameters())
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(batches[b])
        val = evaluate(model, data["validation"])
        hist.append({"epoch": ep + 1, "train loss": total / len(data["train"]), "val f1": val["f1"], "seconds": time.perf_counter() - t0})
        if verbose:
            print(hist[-1])
        if val["f1"] > best_f1:
            best_f1, best_state = val["f1"], copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return hist

In [ ]:
class TokenSoftmax(nn.Module):
    """Baseline：每個字各自 softmax（= 每個字一個邏輯回歸）。"""

    def __init__(self, n_features: int, K: int) -> None:
        super().__init__()
        self.emb = nn.EmbeddingBag(n_features + 1, K, mode="sum", padding_idx=0)
        nn.init.zeros_(self.emb.weight)

    def emissions(self, F: torch.Tensor) -> torch.Tensor:
        B, T, M = F.shape
        return self.emb(F.reshape(B * T, M)).reshape(B, T, -1)

    def loss(self, F, Y, mask):
        E = self.emissions(F)
        return nn.functional.cross_entropy(E[mask], Y[mask], reduction="sum") / len(F)  # 每句的 NLL 加總，再對句平均（跟 CRF 同尺度）

    def predict(self, F, mask):
        E = self.emissions(F)
        return [p[:m].tolist() for p, m in zip(E.argmax(-1), mask.sum(1))]


softmax_model = TokenSoftmax(len(fx_full.index), K)
softmax_hist = train_model(softmax_model, data_full)

In [ ]:
def invalid_transitions(seqs: list[list[int]]) -> int:
    """BIO 不合法的轉移次數：I-X 前面不是 B-X 或 I-X（包括句首就是 I-X）。"""
    bad = 0
    for s in seqs:
        prev = "O"
        for t in s:
            name = label_names[t]
            if name.startswith("I-") and prev[2:] != name[2:]:
                bad += 1
            prev = name
    return bad


results = {}
pred_soft = predict_all(softmax_model, data_full["test"])
results["Token softmax (full features)"] = {**ner_metrics(to_names(test_tags), to_names(pred_soft)), "invalid BIO": invalid_transitions(pred_soft)}
results["(gold labels)"] = {"invalid BIO": invalid_transitions(test_tags)}
display(pd.DataFrame(results).T)

In [ ]:
# TODO 2（問答）
#
# (1) 同樣的特徵，每個字各自 softmax 的 test F1 是多少？比第 20 章的 HMM（≈ 0.73）好多少？好在哪裡（precision 還是 recall）？
# 答：
#
# (2) 不合法的 BIO 轉移有幾個？舉例：「O 後面接 I-PER」代表模型說了什麼？seqeval 會怎麼處理這種預測？
#     為什麼「每個字各自 softmax」沒辦法保證輸出合法？
# 答：

---
## 3. 手寫模型：linear-chain CRF（PyTorch）

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | gold 路徑的分數 | 4 |
| 3B | log-partition（forward）與 Viterbi | 5、6 |
| 3C | 檢查：暴力列舉、邊際機率 = $\partial\ln Z / \partial E$；訓練 | — |
| 3D | 實驗：特徵、轉移矩陣、資料量 | — |

`CRFLayer` 只負責「給定發射分數 $E$，算序列的分數、$\ln Z$、解碼」，不管 $E$ 從哪裡來 —— 3C 用稀疏特徵（`FeatureCRF`），第 4 節換成 BiLSTM（`BiLSTMCRF`），同一個 `CRFLayer`。

### 3A. gold 路徑的分數

$$\text{score}(x, y) = s_{y_1} + e_1(y_1) + \sum_{t=2}^{L}\big[A_{y_{t-1} y_t} + e_t(y_t)\big] + f_{y_L}$$

（$L$ = 這句話的真正長度；padding 的位置不能算進去 → 乘上 mask。）

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $E$ | 發射分數 | $(B, T, K)$ | `E` |
| $A$、$s$、$f$ | 轉移、開始、結束 | $(K, K)$、$(K,)$、$(K,)$ | `self.trans`、`self.start`、`self.end` |
| $e_t(y_t)$ | gold 標籤的發射分數 | $(B, T)$ | `emit` |
| $A_{y_{t-1} y_t}$ | gold 的轉移分數 | $(B, T - 1)$ | `tr` |
| $y_L$ | 每句的最後一個標籤 | $(B,)$ | `last` |

**下標公式**：
- TODO 4：`emit = E.gather(2, Y[:, :, None]).squeeze(2)`（前置複習 8）；`tr = self.trans[Y[:, :-1], Y[:, 1:]]`；
  `score = start[Y[:, 0]] + emit[:, 0] + ((emit[:, 1:] + tr) * mask[:, 1:]).sum(1) + end[last]`，`last = Y.gather(1, (mask.sum(1) − 1)[:, None]).squeeze(1)`。

#### 最小例子：TODO 4、5、6（$K = 2$、$T = 3$）

$E = \begin{pmatrix}1 & 0\\ 0 & 2\\ 1 & 1\end{pmatrix}$（第 $t$ 列 = 第 $t$ 個字對標籤 0、1 的分數），$A = \begin{pmatrix}0.5 & -1\\ -1 & 0.5\end{pmatrix}$（同標籤連續 +0.5、換標籤 −1），$s = f = 0$。

1. **正確答案**：暴力列舉 $2^3 = 8$ 條路徑。例：$(0, 1, 1)$：$1 + (-1 + 2) + (0.5 + 1) = 3.5$。最大是 $(1, 1, 1)$：$0 + (0.5 + 2) + (0.5 + 1) = 4.0$。$\ln Z = \ln\sum e^{\text{score}} \approx 4.8969$。所以 $P(1, 1, 1 \mid x) = e^{4.0 - 4.8969} \approx 0.408$、$P(0, 1, 1 \mid x) \approx 0.247$。
2. **積木**：forward 一步一步：$\alpha_1 = (1, 0)$ → $\alpha_2 \approx (1.5789, 2.9741)$ → $\alpha_3 \approx (3.7210, 4.5279)$ → $\ln Z = \text{logsumexp}(\alpha_3) = 4.8969$。
3. **留給你**：批次 + mask 版（TODO 4、5、6）。

In [ ]:
# 最小例子：TODO 4、5、6（K = 2、T = 3）
E_mini = torch.tensor([[[1.0, 0.0], [0.0, 2.0], [1.0, 1.0]]])  # (B=1, T=3, K=2)
A_mini = torch.tensor([[0.5, -1.0], [-1.0, 0.5]])

# ---- 1. 正確答案：暴力列舉 ----
brute = {}
for path in itertools.product(range(2), repeat=3):
    brute[path] = E_mini[0, 0, path[0]] + sum(A_mini[path[t - 1], path[t]] + E_mini[0, t, path[t]] for t in (1, 2))
    print(path, round(float(brute[path]), 2))
logZ_brute = torch.logsumexp(torch.stack(list(brute.values())), 0)
print("ln Z =", round(float(logZ_brute), 4), "；最佳路徑", max(brute, key=lambda p: brute[p]))

# ---- 2. 積木：forward 一步一步（只有一句、沒有 padding） ----
alpha = E_mini[0, 0]
print("\nalpha_1 =", alpha.numpy().round(4))
for t in (1, 2):
    alpha = torch.logsumexp(alpha[:, None] + A_mini, dim=0) + E_mini[0, t]
    print(f"alpha_{t + 1} =", alpha.numpy().round(4))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 3.5、4.8969、[[1, 1, 1]]）：
# crf_mini = CRFLayer(2); crf_mini.trans.data = A_mini.clone(); m1 = torch.ones(1, 3, dtype=torch.bool); print(round(crf_mini.score(E_mini, torch.tensor([[0, 1, 1]]), m1).item(), 4), round(crf_mini.log_partition(E_mini, m1).item(), 4), crf_mini.decode(E_mini, m1))

In [ ]:
class CRFLayer(nn.Module):
    """linear-chain CRF：轉移、開始、結束分數；給定發射分數 E (B, T, K) 與 mask (B, T)。
    假設每句話的 mask 是「前 L 個 True、後面 False」，且 L ≥ 1。"""

    def __init__(self, K: int) -> None:
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(K, K))
        self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))

    def score(self, E: torch.Tensor, Y: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """gold 路徑的分數，(B,)。"""
        # TODO 4（程式）：下標公式
        raise NotImplementedError("TODO 4：gold 分數")

    def log_partition(self, E: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """ln Z，(B,)。"""
        alpha = self.start + E[:, 0]  # (B, K)
        for t in range(1, E.shape[1]):
            # TODO 5（程式）：
            #   - nxt = logsumexp(alpha[:, :, None] + trans[None] + E[:, t, None, :], dim=1)   # (B, j, k) → 對 j 加總 → (B, k)
            #   - padding 的位置保持不變：alpha = torch.where(mask[:, t, None], nxt, alpha)
            raise NotImplementedError("TODO 5：forward")
        return torch.logsumexp(alpha + self.end, dim=1)

    def nll(self, E: torch.Tensor, Y: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """−ln P(y | x)，batch 平均。"""
        return (self.log_partition(E, mask) - self.score(E, Y, mask)).mean()

    @torch.no_grad()
    def decode(self, E: torch.Tensor, mask: torch.Tensor) -> list[list[int]]:
        """Viterbi：每句話的最佳標籤清單（長度 = 真正的長度）。"""
        B, T, K = E.shape
        delta = self.start + E[:, 0]
        bps = []
        for t in range(1, T):
            # TODO 6（程式）：
            #   - s = delta[:, :, None] + trans[None]（(B, j, k)）；best, bp = s.max(dim=1)
            #   - delta = torch.where(mask[:, t, None], best + E[:, t], delta)
            #   - padding 的位置回溯指標指向自己：bps.append(torch.where(mask[:, t, None], bp, torch.arange(K).expand(B, K)))
            raise NotImplementedError("TODO 6：Viterbi")
        last = (delta + self.end).argmax(1)
        path = [last]
        for bp in reversed(bps):
            last = bp.gather(1, last[:, None]).squeeze(1)
            path.append(last)
        path = torch.stack(path[::-1], dim=1)
        return [p[:L].tolist() for p, L in zip(path, mask.sum(1))]


class FeatureCRF(nn.Module):
    """稀疏特徵的發射分數（同 TokenSoftmax）+ CRFLayer。"""

    def __init__(self, n_features: int, K: int) -> None:
        super().__init__()
        self.emb = nn.EmbeddingBag(n_features + 1, K, mode="sum", padding_idx=0)
        nn.init.zeros_(self.emb.weight)
        self.crf = CRFLayer(K)

    def emissions(self, F: torch.Tensor) -> torch.Tensor:
        B, T, M = F.shape
        return self.emb(F.reshape(B * T, M)).reshape(B, T, -1)

    def loss(self, F, Y, mask):
        return self.crf.nll(self.emissions(F), Y, mask)

    def predict(self, F, mask):
        return self.crf.decode(self.emissions(F), mask)

### 3C. 檢查與訓練

1. **暴力列舉**：隨機的參數、幾句截短到 4 個字的句子（$9^4 = 6{,}561$ 條路徑），比 $\ln Z$ 與 Viterbi；同一個 batch 裡放不同長度，檢查 mask。
2. **邊際機率 = 梯度**（前置複習 4）：$\partial\ln Z / \partial e_t(k) = P(y_t = k \mid x)$。用 autograd 算 $\ln Z$ 對 $E$ 的梯度，跟暴力列舉的邊際機率比。
3. **訓練**：Adam、學習率 0.05、L2 $10^{-6}$、8 個 epoch，留 validation 最好的。

In [ ]:
torch.manual_seed(1)
crf_chk = CRFLayer(K)
with torch.no_grad():
    crf_chk.trans.normal_()
    crf_chk.start.normal_()
    crf_chk.end.normal_()
lens_chk = [4, 2, 3]
E_chk = torch.randn(3, 4, K, requires_grad=True)
mask_chk = torch.tensor([[t < L for t in range(4)] for L in lens_chk])
logZ = crf_chk.log_partition(E_chk, mask_chk)
dec = crf_chk.decode(E_chk.detach(), mask_chk)
marg = torch.autograd.grad(logZ.sum(), E_chk)[0]  # (B, T, K)：∂ ln Z / ∂ E
for b, L in enumerate(lens_chk):
    paths = list(itertools.product(range(K), repeat=L))
    Yb = torch.tensor([list(p) + [0] * (4 - L) for p in paths])
    sc = crf_chk.score(E_chk[b : b + 1].detach().expand(len(paths), -1, -1), Yb, mask_chk[b : b + 1].expand(len(paths), -1))
    probs = torch.softmax(sc, 0)
    brute_marg = torch.zeros(L, K)
    for p, pr in zip(paths, probs):
        brute_marg[torch.arange(L), torch.tensor(p)] += pr
    print(f"句 {b}（長度 {L}，{len(paths):,} 條路徑）：✅ ln Z 差 {abs(logZ[b].item() - torch.logsumexp(sc, 0).item()):.1e}；"
          f"Viterbi 一樣？{dec[b] == list(paths[int(sc.argmax())])}；邊際機率最大差 {float((marg[b, :L] - brute_marg).abs().max()):.1e}；padding 位置的梯度 = 0？{bool((marg[b, L:] == 0).all())}")

In [ ]:
crf_model = FeatureCRF(len(fx_full.index), K)
crf_hist = train_model(crf_model, data_full)
pred_crf = predict_all(crf_model, data_full["test"])
results["CRF scratch (full features)"] = {**ner_metrics(to_names(test_tags), to_names(pred_crf)), "invalid BIO": invalid_transitions(pred_crf)}
display(pd.DataFrame(results).T)

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, h in [("token softmax", softmax_hist), ("CRF", crf_hist)]:
    ax.plot([r["epoch"] for r in h], [r["val f1"] for r in h], "o-", label=name)
ax.set_xlabel("epoch")
ax.set_ylabel("validation F1")
ax.legend()
plt.show()

In [ ]:
# TODO 7（問答）TODO 3–6 與訓練
#
# (1) 最小例子：
#     (a) 為什麼 (1, 1, 1) 的分數最高，雖然第 1 個字對標籤 0 的分數比較高？轉移分數 A 扮演什麼角色？
#     (b) P(1, 1, 1 | x) 只有 0.41 —— 剩下的機率在哪裡？
# 答：
#
# (2) CRF 的 forward 跟第 20 章 HMM 的 forward 有哪裡不一樣？（提示：HMM 的 A、B 是機率，每一列加總 1；CRF 的是任意實數）
#     為什麼 CRF 要除以 Z(x)，HMM 卻不用？
# 答：
#
# (3) 邊際機率 = ∂ ln Z / ∂E 的檢查通過了。用這個解釋：NLL 對 e_t(k) 的梯度是什麼？什麼時候梯度是 0？（對照第三章邏輯回歸）
# 答：
#
# (4) mask：padding 的位置梯度是 0 —— 如果忘了 torch.where，ln Z 會多算什麼？
# 答：
#
# (5) 訓練：CRF 和 token softmax 的 test F1 差多少？不合法的 BIO 轉移有幾個？validation F1 在第幾個 epoch 最好、之後下降了嗎？
# 答：

### 3D. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | 學到的轉移矩陣長什麼樣子？哪些轉移是大負數？ |
| 2 | 特徵消融：只有字 → +字形 → +字首字尾 → +前後文，F1 怎麼變？CRF 和 token softmax 的差距跟著怎麼變？ |
| 3 | 訓練資料量：10%、30%、100% 的訓練句子 |

In [ ]:
# 實驗 1：轉移矩陣
fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={"width_ratios": [4, 1]})
sns.heatmap(crf_model.crf.trans.detach().numpy(), annot=True, fmt=".1f", cmap="coolwarm", center=0, xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_title("learned transition scores A (row = from, column = to)")
sns.heatmap(torch.stack([crf_model.crf.start, crf_model.crf.end], 1).detach().numpy(), annot=True, fmt=".1f", cmap="coolwarm", center=0, xticklabels=["start", "end"], yticklabels=label_names, ax=axes[1])
plt.show()

In [ ]:
# 實驗 2：特徵消融（每組 6 個 epoch）
abl_rows = []
for name, groups in FEATURE_SETS.items():
    fx, data = (fx_full, data_full) if name == "full (+context)" else encode_splits(groups)
    for model_name, cls in [("token softmax", TokenSoftmax), ("CRF", FeatureCRF)]:
        if name == "full (+context)":
            m = softmax_model if cls is TokenSoftmax else crf_model
        else:
            m = cls(len(fx.index), K)
            train_model(m, data, epochs=6, verbose=False)
        r = evaluate(m, data["test"])
        abl_rows.append({"features": name, "model": model_name, "# feature strings": len(fx.index), "test f1": r["f1"], "precision": r["precision"], "recall": r["recall"]})
abl = pd.DataFrame(abl_rows)
display(abl.pivot(index="features", columns="model", values="test f1").loc[list(FEATURE_SETS)].assign(**{"CRF − softmax": lambda d: d["CRF"] - d["token softmax"]}).round(4))

In [ ]:
# 實驗 3：訓練資料量（完整特徵；特徵表用全部 train 建，但只用部分句子訓練）
size_rows = []
for frac in [0.1, 0.3, 1.0]:
    if frac == 1.0:
        m = crf_model
    else:
        idx = np.random.default_rng(0).permutation(len(data_full["train"]))[: int(frac * len(data_full["train"]))]
        sub = {**data_full, "train": [data_full["train"][i] for i in idx]}
        m = FeatureCRF(len(fx_full.index), K)
        train_model(m, sub, epochs=8, verbose=False)
    size_rows.append({"train fraction": frac, "# sentences": int(frac * len(data_full["train"])), **evaluate(m, data_full["test"])})
display(pd.DataFrame(size_rows).set_index("train fraction").round(4))

In [ ]:
# TODO 8（問答）實驗 1–3
#
# (1) 轉移矩陣：
#     (a) O → I-PER、B-PER → I-ORG 的分數是多少？為什麼是大負數？（對照 EDA：train 裡出現過幾次？）
#     (b) start 列：哪些標籤當句首的分數最低？
#     (c) 第 20 章 HMM 的轉移是機率（log10 ≈ −4），這裡是分數 —— 意義上有什麼差別？
# 答：
#
# (2) 特徵消融：
#     (a) 哪一組特徵進步最多？為什麼？（提示：OOV 的實體字 —— 只有 "w=" 的時候，沒看過的字只剩 bias）
#         有沒有哪一組「加了反而變差」？可能的原因？（提示：特徵變多 → 更容易過擬合；這裡只訓練 6 個 epoch、L2 沒有調）
#     (b) 「CRF − softmax」的差距在只有字的時候大、還是有前後文的時候大？為什麼前後文特徵會「取代」一部分轉移分數的功能？
# 答：
#
# (3) 資料量：10% 的資料（約 1,400 句）F1 是多少？跟第 20 章用 100% 資料的 HMM 比呢？判別式模型 + 好的特徵 vs 生成式模型 + 更多資料，你學到什麼？
# 答：

---
## 4. 套件實作：`sklearn-crfsuite`，與 BiLSTM-CRF

### `sklearn-crfsuite`（CRFsuite 的 Python 包裝）

| 參數 | 意思 |
| --- | --- |
| `algorithm="lbfgs"` | 二階近似的批次最佳化（整份資料算梯度），比我們的 Adam 穩定 |
| `c1`、`c2` | L1、L2 正則化的強度（L1 會讓很多特徵權重變成 0） |
| `max_iterations` | L-BFGS 的迭代次數 |
| `all_possible_transitions` | 也學「train 裡沒出現過的轉移」（給它們負分數） |

輸入格式：每個字一個 `dict`（特徵名稱 → 值），我們把特徵字串清單轉成 `{特徵: 1}`。

### BiLSTM-CRF（Lample et al. 2016）

把「稀疏特徵加總」換成 **BiLSTM**：字的向量（embedding）+ 字形的向量 → 雙向 LSTM（從左讀、從右讀，各自記住前後文）→ 線性層 → 發射分數 $E$ → **同一個 `CRFLayer`**。不用手工設計前後文特徵，LSTM 自己學。（LSTM 的細節在第三部分；這裡先當成「看得到整句話的特徵產生器」。）

| 參數 | 本章的值 |
| --- | --- |
| 字的 embedding | 50 維；train 出現 ≥ 2 次的小寫字，其他 → UNK |
| 字形的 embedding | 16 維 |
| LSTM 隱藏層 | 每個方向 100 |

In [ ]:
import sklearn_crfsuite


def to_crfsuite(toks_list: list[list[str]]) -> list[list[dict]]:
    groups = FEATURE_SETS["full (+context)"]
    return [[{f: 1.0 for f in token_features(s, i, groups)} for i in range(len(s))] for s in toks_list]


t0 = time.perf_counter()
crfsuite = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=100, all_possible_transitions=True)
crfsuite.fit(to_crfsuite(train_tokens), to_names(train_tags))
t_suite = time.perf_counter() - t0
pred_suite = crfsuite.predict(to_crfsuite(test_tokens))
results["sklearn-crfsuite (L-BFGS, full features)"] = {**ner_metrics(to_names(test_tags), pred_suite), "invalid BIO": invalid_transitions([[label_names.index(t) for t in s] for s in pred_suite])}
print(f"sklearn-crfsuite 訓練 {t_suite:.0f} 秒；非零的特徵權重：{len(crfsuite.state_features_):,}（L1 把其他的變成 0）")
display(pd.DataFrame(results).T)

In [ ]:
class BiLSTMCRF(nn.Module):
    """字 embedding + 字形 embedding → BiLSTM → Linear → CRFLayer。
    輸入的 F 是 (B, T, 2)：[字的編號, 字形的編號]，0 = padding。"""

    def __init__(self, n_words: int, n_shapes: int, K: int, word_dim: int = 50, shape_dim: int = 16, hidden: int = 100) -> None:
        super().__init__()
        self.word = nn.Embedding(n_words, word_dim, padding_idx=0)
        self.shape = nn.Embedding(n_shapes, shape_dim, padding_idx=0)
        self.lstm = nn.LSTM(word_dim + shape_dim, hidden, batch_first=True, bidirectional=True)
        self.out = nn.Linear(2 * hidden, K)
        self.crf = CRFLayer(K)

    def emissions(self, F: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        h = torch.cat([self.word(F[:, :, 0]), self.shape(F[:, :, 1])], dim=-1)
        packed = nn.utils.rnn.pack_padded_sequence(h, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=F.shape[1])
        return self.out(out)

    def loss(self, F, Y, mask):
        return self.crf.nll(self.emissions(F, mask), Y, mask)

    def predict(self, F, mask):
        return self.crf.decode(self.emissions(F, mask), mask)


lower_count = Counter(w.lower() for s in train_tokens for w in s)
word_index = {w: i + 2 for i, w in enumerate(sorted(w for w, c in lower_count.items() if c >= 2))}  # 0 = padding、1 = UNK
shape_index = {s: i + 2 for i, s in enumerate(sorted({word_shape(w) for s_ in train_tokens for w in s_}))}
data_lstm = {name: [(torch.tensor([[word_index.get(w.lower(), 1), shape_index.get(word_shape(w), 1)] for w in s]), torch.tensor(y)) for s, y in zip(toks, tags)]
             for name, (toks, tags) in splits.items()}
print(f"字表 {len(word_index):,}（+ padding、UNK）；字形 {len(shape_index)} 種")

t0 = time.perf_counter()
lstm_model = BiLSTMCRF(len(word_index) + 2, len(shape_index) + 2, K)
lstm_hist = train_model(lstm_model, data_lstm, epochs=6, lr=0.005, l2=0.0)
t_lstm = time.perf_counter() - t0
pred_lstm = predict_all(lstm_model, data_lstm["test"])
results["BiLSTM-CRF (scratch CRF layer)"] = {**ner_metrics(to_names(test_tags), to_names(pred_lstm)), "invalid BIO": invalid_transitions(pred_lstm)}
print(f"BiLSTM-CRF 訓練 {t_lstm:.0f} 秒；參數 {sum(p.numel() for p in lstm_model.parameters()):,} 個")
display(pd.DataFrame(results).T)

In [ ]:
# TODO 9（問答）
#
# (1) sklearn-crfsuite 跟手寫的 CRF（同樣的特徵）F1 差多少？兩者的差別在哪裡？（最佳化方法、L1、all_possible_transitions）
# 答：
#
# (2) BiLSTM-CRF：
#     (a) 沒有手工的字首字尾、前後文特徵，F1 跟特徵 CRF 比如何？它從哪裡得到前後文？
#     (b) 訓練時間、參數量差多少？
#     (c) BiLSTM-CRF 也有一些不合法的 BIO 轉移 —— CRF 層為什麼沒有完全擋住？（提示：轉移分數只是「分數」不是硬性限制；
#         看 lstm_model.crf.trans 的 O → I-X 分數，跟特徵 CRF 的比；l2 = 0、只訓練 6 個 epoch）
#     (d) 如果把 BiLSTM 換成預訓練的 BERT（第五部分），你預期 F1 會怎麼變？為什麼？（提示：OOV 的人名、地名）
# 答：

---
## 5. 評估

- 實體層級的 precision／recall／F1（`ner_metrics`，seqeval）—— 主要指標是 **test F1**；
- 各實體類型的 F1（`seqeval.metrics.classification_report`）；
- 成績寫進 `results/conll2003_ner.csv`，跟第 20 章的 HMM 一起比。

In [ ]:
from seqeval.metrics import classification_report
from ml_dl_llm.results import compare, log_result

BENCH = "conll2003_ner"
for name, m in results.items():
    if name.startswith("("):
        continue
    log_result(BENCH, name, {k: m[k] for k in ["precision", "recall", "f1", "token_accuracy"]}, split="test")
compare(BENCH, "f1")

per_type = {}
for name, pred in [("token softmax", to_names(pred_soft)), ("CRF scratch", to_names(pred_crf)), ("crfsuite", pred_suite), ("BiLSTM-CRF", to_names(pred_lstm))]:
    rep = classification_report(to_names(test_tags), pred, output_dict=True)
    per_type[name] = {k: rep[k]["f1-score"] for k in ["PER", "ORG", "LOC", "MISC"]}
display(pd.DataFrame(per_type).round(4))

In [ ]:
# TODO 10（問答）評估
#
# (1) 依 test F1 排序：HMM、token softmax、CRF（手寫）、crfsuite、BiLSTM-CRF。最大的進步來自哪一步（換成判別式＋特徵、加上轉移、換成 BiLSTM）？
# 答：
#
# (2) 各實體類型：哪一類最難？為什麼 MISC、ORG 比 PER、LOC 難？
# 答：
#
# (3) validation F1 和 test F1 差多少？（CoNLL-2003 的 test 是「比較晚的新聞」，出現很多 train 沒有的名字）這叫什麼問題？
# 答：

---
## 6. 診斷：CRF 錯在哪裡？

跟第 20 章一樣，依「OOV」與「實體類型」拆開；再看實體層級的錯誤種類：

| 錯誤種類 | 定義 |
| --- | --- |
| 漏掉（missed） | gold 的實體，預測裡完全沒有重疊的實體 |
| 多標（spurious） | 預測的實體，gold 裡完全沒有重疊的 |
| 類型錯（wrong type） | 範圍一樣、類型不同 |
| 邊界錯（boundary） | 有重疊，但範圍不一樣 |

In [ ]:
def error_types(gold_seqs: list[list[str]], pred_seqs: list[list[str]]) -> Counter:
    c = Counter()
    for g, p in zip(gold_seqs, pred_seqs):
        G, P = set(entities(g)), set(entities(p))
        for e in G - P:
            over = [q for q in P if q[1] < e[2] and e[1] < q[2]]
            if not over:
                c["missed"] += 1
            elif any(q[1:] == e[1:] for q in over):
                c["wrong type"] += 1
            else:
                c["boundary"] += 1
        c["spurious"] += sum(not any(q[1] < e[2] and e[1] < q[2] for e in G) for q in P - G)
        c["correct"] += len(G & P)
    return c


err = pd.DataFrame({name: error_types(to_names(test_tags), pred) for name, pred in
                    [("token softmax", to_names(pred_soft)), ("CRF scratch", to_names(pred_crf)), ("BiLSTM-CRF", to_names(pred_lstm))]}).fillna(0).astype(int)
display(err)

gold_flat = np.concatenate(test_tags)
pred_flat = np.concatenate(pred_crf)
oov_flat = np.array([word_count[w] == 0 for s in test_tokens for w in s])
diag = pd.DataFrame({"type": [label_names[t][2:] if t else "O" for t in gold_flat], "correct": gold_flat == pred_flat, "OOV": oov_flat})
display(diag.groupby(["type", "OOV"])["correct"].mean().unstack().round(3))

shown = 0
for s, g, p in zip(test_tokens, test_tags, pred_crf):
    if g != p and 6 <= len(s) <= 16 and shown < 5:
        print(" ".join(f"{w}/{label_names[a]}" + ("" if a == b else f"→{label_names[b]}") for w, a, b in zip(s, g, p)))
        shown += 1

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 錯誤種類：CRF 比 token softmax 少了哪一種錯誤最多？用轉移分數解釋。
# 答：
#
# (2) OOV：CRF 在 OOV 實體字的準確率比第 20 章的 HMM 好多少？哪些特徵讓它能處理沒看過的字？
# 答：
#
# (3) 看錯誤的句子：剩下的錯誤需要什麼資訊才能修正？（世界知識？更遠的前後文？同一篇文章裡前面出現過的全名？）
# 答：
#
# (4) 第一部分（經典機器學習）總結：HMM、CRF 跟前面哪些模型是「同一個想法的序列版」？
#     |              | 獨立樣本的版本 | 序列的版本 |
#     | 生成式        |               | HMM        |
#     | 判別式        |               | CRF        |
#     | 隱藏變數 + EM |               | Baum-Welch |
# 答：

---
## 7. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

CoNLL-2003 NER（test；實體層級）

| 方法 | 做法 | precision | recall | F1 | 不合法 BIO |
| --- | --- | --- | --- | --- | --- |
| HMM（第 20 章） | 數數 + Viterbi | | | | |
| token softmax | 稀疏特徵，每個字各自 softmax | | | | |
| CRF（手寫） | 稀疏特徵 + 轉移，整條序列正規化 | | | | |
| sklearn-crfsuite | 同樣的特徵，L-BFGS + L1/L2 | | | | |
| BiLSTM-CRF | 字／字形 embedding + BiLSTM + 手寫 CRF 層 | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ CRF 是無向圖模型；因子圖、一般圖結構的 CRF（loopy belief propagation）
- ☐ MEMM 與 label bias 問題（為什麼要「整條序列」正規化而不是每一步正規化）
- ☐ 結構化感知器、結構化 SVM（另一種訓練序列模型的方法）
- ☐ 用 forward-backward 手算梯度（不靠 autograd）

**延伸**
- ☐ 字元層級的 CNN／LSTM 特徵（處理 OOV）；預訓練詞向量（GloVe）
- ☐ BERT + CRF（第五部分 BERT）；用 subword 時的標籤對齊
- ☐ 巢狀 NER、span-based NER
- ☐ 中文 NER（沒有空白分詞）；詞性標註、chunking（CoNLL-2003 也有這兩種標籤）